<center><p float="center">
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<h1><center> Real-Time Retail Feedback Intelligence
 </center></h1>


### **Business Context**
Why is this problem important to solve?

### **Objective**

What is the intended goal?
### **Dataset Used for the Notebook**
Describe dataset used for this project.

### **Installing and Importing Necessary Libraries**
First, let's set up the environment by installing the required Python libraries.

In [ ]:
# Install the required libraries for the project
# --- Installation of Required Libraries ---
!pip install pandas matplotlib seaborn
!pip install -q -U google-genai
!pip install openai
!pip install wordcloud
!pip install plotly


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.5/260.5 kB 8.3 MB/s eta 0:00:00


In [ ]:
# Import the required libraries for the project
# --- Core Utilities ---
import time
import json
import re
import numpy as np
from typing import Dict, List, Optional, Any, Tuple

# --- Data Handling ---
import pandas as pd
from google.colab import userdata

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display

# --- Machine Learning Metrics ---
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# --- Progress Bar ---
from tqdm import tqdm

# --- OpenAI Client ---
import openai

# Set global plot style
sns.set_style("whitegrid")

### **Data Loading**
### Loading and Understanding the Data


In [ ]:
### Loading and Understanding the Data
# Load the dataset from 'data.csv'. Remember that the separator is a semicolon (';')
# and the first column should be used as the index.

df = pd.read_csv(_________, sep=_________, index_col=_________)
display(df)

,Clothing.ID,Age,Title,Review.Text,Rating,Recommended.IND,Positive.Feedback.Count,Division.Name,Department.Name,Class.Name
1,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
2,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
3,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
4,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
5,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses
...,...,...,...,...,...,...,...,...,...,...
23482,1104,34,Great dress for many occasions,I was very happy to snag this dress at such a ...,5,1,0,General Petite,Dresses,Dresses
23483,862,48,Wish it was made of cotton,"It reminds me of maternity clothes. soft, stre...",3,1,0,General Petite,Tops,Knits
23484,1104,31,"Cute, but see through","This fit well, but the top was very see throug...",3,0,1,General Petite,Dresses,Dresses
23485,1084,28,"Very cute dress, perfect for summer parties an...",I bought this dress for a wedding i have this ...,3,1,2,General,Dresses,Dresses


### **Data Overview**

In [ ]:
# Get a summary of the dataset to check data types and non-null counts
print("\nDataset Info:")
df.info()

Dataset Head:


,Clothing.ID,Age,Title,Review.Text,Rating,Recommended.IND,Positive.Feedback.Count,Division.Name,Department.Name,Class.Name
1,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
2,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
3,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
4,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
5,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


### **Sanity checks**

In [ ]:
# Check for the total number of missing values in each column
print("\nMissing Values:")
print(df.isnull().sum())



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
Index: 23486 entries, 1 to 23486
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Clothing.ID              23486 non-null  int64 
 1   Age                      23486 non-null  int64 
 2   Title                    19676 non-null  object
 3   Review.Text              22641 non-null  object
 4   Rating                   23486 non-null  int64 
 5   Recommended.IND          23486 non-null  int64 
 6   Positive.Feedback.Count  23486 non-null  int64 
 7   Division.Name            23472 non-null  object
 8   Department.Name          23472 non-null  object
 9   Class.Name               23472 non-null  object
dtypes: int64(5), object(5)
memory usage: 2.0+ MB


### **Data Cleaning and Preprocessing**

**Think about it:** The Review Text column is the most critical feature for our Generative AI model. What should be done with rows where this text is missing?

In [ ]:
df = df._______

print("\nMissing Values after cleaning:")
print(df.isnull().sum())

### **Exploratory Data Analysis**

EDA is an important part of any project involving data. It is important to investigate and understand the data better before building a model with it. A few questions have been mentioned below which will help you approach the analysis in the right manner and generate insights from the data. A thorough analysis of the data, in addition to the questions mentioned below, should be done.


In [ ]:
## 4.1 Overall Sentiment: Distribution of Product Ratings
# Create a histogram using Plotly Express to visualize the distribution of the 'Rating' column.
# This will help you understand the overall customer sentiment.

fig = px.histogram(_________, x="Rating", color="Rating",
                   title="Distribution of Product Ratings")

fig.update_layout(
    xaxis_title="Rating (1-5)",
    yaxis_title="Number of Reviews"
)
fig.show()

In [ ]:
## 4.2 Performance by Category: Average Rating by Department
# Calculate the average rating for each department.
# Group the DataFrame by 'Department.Name' and calculate the mean of the 'Rating'.

avg_rating_by_dept = df.groupby(_________)[_________].mean().sort_values(ascending=False).reset_index()

# Create a horizontal bar chart to show the average rating for each department.
fig = px.bar(avg_rating_by_dept,
             x='Rating',
             y='Department.Name',
             orientation='h',
             title="Average Customer Rating by Department",
             color='Rating')
fig.show()

In [ ]:
## 4.3 Qualitative Insights: Word Clouds for Positive vs. Negative Reviews
# Separate the reviews into two strings: one for positive (Rating >= 4) and one for negative (Rating <= 2).
positive_reviews = " ".join(df[df['Rating'] >= 4]['Review.Text'])
negative_reviews = " ".join(df[_________]['Review.Text'])

# Generate a WordCloud for positive reviews
wordcloud_positive = WordCloud(width=800, height=400, background_color='white').generate(_________)

# Generate a WordCloud for negative reviews
wordcloud_negative = WordCloud(width=800, height=400, background_color='black', colormap='Reds').generate(_________)

# Plot the word clouds side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
ax1.imshow(wordcloud_positive, interpolation='bilinear')
ax1.set_title('Most Common Words in Positive Reviews', fontsize=20)
ax1.axis('off')
ax2.imshow(wordcloud_negative, interpolation='bilinear')
ax2.set_title('Most Common Words in Negative Reviews', fontsize=20)
ax2.axis('off')
plt.show()

## **Building the Generative AI Pipeline**

We will now build a system to analyze the reviews. This involves setting up the AI client, designing prompts, generating structured data, and evaluating the results. Use gpt-4o-mini model

#### **Setup AI Client and Data Sample**

**Questions:**

1.  How do you initialize the OpenAI client with your API key and the correct base URL?
    

In [ ]:
# --- OpenAI API Setup ---
# Fetch your stored API key.
api_key = userdata.get('OPENAI_API_KEY')

# Initialize the OpenAI client. Provide your API key and the custom base URL.
client = openai.OpenAI(
    api_key=_________,
    base_url="https://aibe.mygreatlearning.com/openai/v1"
)

model_name = __________



#### **Note:**

For this project, we will analyze and categorize a sample of **50 customer reviews**. This number is chosen intentionally. Since the API has a **budget limit of $20**, running prompts on very large datasets can quickly exhaust your quota—especially because this exercise may involve **multiple iterations, prompt refinements, and repeated evaluations**.

To avoid unnecessary cost and ensure efficient experimentation, we recommend the following approach:

*   **Use very small samples (5–10 reviews)** during the **initial testing phase** to validate your prompt structure and logic.
    
*   **Scale up to 50 reviews** for the **final evaluation phase**, ensuring you get enough data to compare prompting techniques without draining your budget.
    
*   This strategy helps maintain cost control while still providing meaningful insights across Zero-Shot, Few-Shot, and Chain-of-Thought techniques.
    

If your API quota gets exhausted, you may temporarily switch to another Free AI assistant API. However, note that external tools may also have **rate limits** or **token caps**, so you will need to build retry logic and manage throttling within your code.

In [ ]:
# Create a sample of 50 reviews to work with.
df_sample_copy = ______________

#### **Prompt Engineering and Evaluation**

We will test three different prompting techniques. For each, we will create a basic version (V1) and an enhanced version (V2).

**Think about it:** Why is it important to have a consistent and robust evaluation framework? How can we use an "LLM-as-Judge" to score the quality of our generated outputs objectively?

#### **Technique 1: Zero-Shot Prompting**

**Questions:**

1.  How would you design a basic Zero-Shot prompt that asks the model for Category, Sentiment, Summary, Personalized Message, and Retail Insight?
    
2.  How can you enhance this prompt with more business context (e.g., a company name, the importance of accuracy) to create a V2 prompt?
    
3.  How will you loop through the data sample to generate and store the structured output for both prompt versions?
    
4.  How will you apply the LLM-as-Judge to generate a evaluation score between 0 to 1 (decimal allowed) for the outputs and calculate the average score of Version 1 and Version 2 prompt?

**How the process works:**

1.  First, you create an **LLM-as-a-judge** function that can evaluate the quality of model outputs.
    
2.  Then, you run your **Zero-Shot Prompt Version 1** on a sample of 50 reviews to generate predictions.
    
3.  You use the judge function to **score each prediction** and compute the **average score for Version 1**.
    
4.  Next, you repeat the same workflow with your **Version 2 prompt**, generate predictions, evaluate them, and calculate the **average score for Version 2**.

In [ ]:
# --- Generating LLM Predictions ---
def get_structured_output(review_text: str, prompt: str) -> dict:
    try:
        # Make an API call to the chat completions endpoint.
        # Pass the model name, temperature, and messages (system and user roles).
        resp = client.chat.completions.create(
            model=_________,
            temperature=_________,
            messages=[
                {"role": "system", "content": "You are a helpful AI assistant."},
                {"role": "user", "content": f"{prompt}\n\nReview: {review_text}"}
            ]
        )
    except Exception as e:
        return {"Summary": f"ERROR_CALL: {type(e).__name__}: {e}"}

    text = resp.choices[0].message.content or ""
    lines = [ln.strip() for ln in text.strip().splitlines() if ln.strip()]
    out = {"Category": "", "Sentiment": "", "Summary": "", "Personalized_Message": "", "Retail_Insight": ""}
    for line in lines:
        if ":" in line:
            key_raw, val = line.split(":", 1)
            key = key_raw.strip().lower()
            val = val.strip()
            if key.startswith("category"): out["Category"] = val
            elif key.startswith("sentiment"): out["Sentiment"] = val
            elif key.startswith("summary"): out["Summary"] = val
            elif "personal" in key: out["Personalized_Message"] = val
            elif "insight" in key: out["Retail_Insight"] = val
    return out


In [ ]:
# --- Prompting Strategies and Evaluation Framework ---
def generate_output(review_text, prompt):
    return get_structured_output(review_text, prompt)

# The judge_prompt is provided for you. It contains the rules for the LLM evaluator.
judge_prompt = ____________

def judge_output(output_text):
    try:
        # Call the LLM with the judge_prompt to get a score.
        resp = client.chat.completions.create(
            model=model_name,
            temperature=0,
            messages=[
                {"role": "system", "content": "You are a helpful AI assistant."},
                {"role": "user", "content": judge_prompt.format(output=output_text)}
            ]
        )
        score_str = resp.choices[0].message.content.strip()
        return float(score_str)
    except Exception:
        return None

In [ ]:
print("Zero-Shot Version 1 predictions...")
df_sample = df_sample_copy.copy()

# Define the first version of your Zero-Shot prompt.
zero_shot_prompt_v1 = _______


# Loop through each review in the sample, call the generate_output function, and store the results.
zs_list = [generate_output(_________, _________) for review in tqdm(df_sample["Review.Text"], desc="Zero-Shot Predictions")]
zs_df = pd.DataFrame(zs_list)
df_sample = pd.concat([df_sample, zs_df.add_prefix("ZS_V1_")], axis=1)



In [ ]:
print("Judging Zero-Shot V1 outputs...")
# Loop through the generated summaries and use the judge_output function to score them.
zs_scores = [judge_output(_________) for output in tqdm(df_sample["ZS_V1_Summary"], desc="Judge ZS_V1")]
df_sample["ZS_V1_Score"] = zs_scores
print(f"Avg ZS_V1_Score: {df_sample['ZS_V1_Score'].mean():.3f}")

In [ ]:
print("\nZero-Shot V2 predictions (context-enhanced)...")

# Define the second version of your Zero-Shot prompt.
zero_shot_prompt_v2 = ________

# Loop through each review in the sample, call the generate_output function, and store the results.
zsv2_list = [generate_output(_________, _________) for review in tqdm(df_sample["Review.Text"], desc="Zero-Shot V2 Predictions")]
zsv2_df = pd.DataFrame(zsv2_list)
df_sample = pd.concat([df_sample, zsv2_df.add_prefix("ZS_V2_")], axis=1)



In [ ]:
print("Judging Zero-Shot V2 outputs...")
# Loop through the generated summaries and use the judge_output function to score them.
zsv2_scores = [judge_output(_________) for output in tqdm(df_sample["ZS_V2_Summary"], desc="Judge ZS_V2")]
df_sample["ZS_V2_Score"] = zsv2_scores
print(f"Avg ZS_V2_Score: {df_sample['ZS_V2_Score'].mean():.3f}")

#### **Technique 2: Few-Shot Prompting**

**Questions:**

1.  How do you structure a Few-Shot prompt? What kind of examples (e.g., one positive, one negative) would be most effective?
    
2.  For the V2 prompt, how can you add a set of "rules" to guide the model's output for each field, reducing ambiguity?
    
3.  After generating and scoring the outputs, how does the performance of Few-Shot prompting compare to previous version?

**How the process works:**

1.  First, you create an **LLM-as-a-judge** function that can evaluate the quality of model outputs.
    
2.  Then, you run your **Prompt Version 1** on a sample of 50 reviews to generate predictions.
    
3.  You use the judge function to **score each prediction** and compute the **average score for Version 1**.
    
4.  Next, you repeat the same workflow with your **Version 2 prompt**, generate predictions, evaluate them, and calculate the **average score for Version 2**.

In [ ]:
# --- Technique 2: Few-Shot Prompting ---

print("\nFew-Shot V1 predictions...")
# Define the first version of your Few-Shot prompt.
few_shot_prompt_v1 = ______

# Loop through each review in the sample, call the generate_output function, and store the results.
fs_list = [generate_output(_________, _________) for review in tqdm(df_sample["Review.Text"], desc="Few-Shot V1 Predictions")]
fs_df = pd.DataFrame(fs_list)
df_sample = pd.concat([df_sample, fs_df.add_prefix("FS_V1_")], axis=1)


In [ ]:
print("Judging Few-Shot V1 outputs...")
# Loop through the generated summaries and use the judge_output function to score them.
fs_v1_scores = [judge_output(_________) for output in tqdm(df_sample["FS_V1_Summary"], desc="Judge FS_V1")]
df_sample["FS_V1_Score"] = fs_v1_scores
print(f"Avg FS_V1_Score: {df_sample['FS_V1_Score'].mean():.3f}")


In [ ]:
print("\nFew-Shot V2 predictions (context-enhanced)...")
# Define the second version of your few-Shot prompt.
few_shot_prompt_v2 = _______

# Loop through each review in the sample, call the generate_output function, and store the results.
fs_v2_list = [generate_output(_________, _________) for review in tqdm(df_sample["Review.Text"], desc="Few-Shot V2 Predictions")]
fs_v2_df = pd.DataFrame(fs_v2_list)
df_sample = pd.concat([df_sample, fs_v2_df.add_prefix("FS_V2_")], axis=1)



In [ ]:
print("Judging Few-Shot V2 outputs...")
# Loop through the generated summaries and use the judge_output function to score them.
fs_v2_scores = [judge_output(_________) for output in tqdm(df_sample["FS_V2_Summary"], desc="Judge FS_V2")]
df_sample["FS_V2_Score"] = fs_v2_scores
print(f"Avg FS_V2_Score: {df_sample['FS_V2_Score'].mean():.3f}")

#### **Technique 3: Chain-of-Thought (CoT) Prompting**

**Questions:**

1.  How do you instruct the model to "think step-by-step" internally but only show the final, structured answer?
    
2.  How can you combine the CoT instruction with more detailed reasoning from the COT V1 prompt to create a powerful CoT V2 prompt?
    
3.  Does encouraging the model to reason first lead to a measurable improvement in the quality of the generated insights?

**How the process works:**

1.  First, you create an **LLM-as-a-judge** function that can evaluate the quality of model outputs.
    
2.  Then, you run your **Prompt Version 1** on a sample of 50 reviews to generate predictions.
    
3.  You use the judge function to **score each prediction** and compute the **average score for Version 1**.
    
4.  Next, you repeat the same workflow with your **Version 2 prompt**, generate predictions, evaluate them, and calculate the **average score for Version 2**.

In [ ]:
# --- Technique 3: Chain-of-Thought (CoT) Prompting ---
print("\nCoT V1 predictions...")

# Define the first version of your COT prompt.
cot_prompt_v1 = __________

# Loop through each review in the sample, call the generate_output function, and store the results.
cot_list = [generate_output(_________, _________) for review in tqdm(df_sample["Review.Text"], desc="CoT V1 Predictions")]
cot_df = pd.DataFrame(cot_list)
df_sample = pd.concat([df_sample, cot_df.add_prefix("COT_V1_")], axis=1)


In [ ]:
print("Judging CoT V1 outputs...")
# Loop through the generated summaries and use the judge_output function to score them.
cot_v1_scores = [judge_output(_________) for output in tqdm(df_sample["COT_V1_Summary"], desc="Judge COT_V1")]
df_sample["COT_V1_Score"] = cot_v1_scores
print(f"Avg COT_V1_Score: {df_sample['COT_V1_Score'].mean():.3f}")

In [ ]:
print("\nCoT V2 predictions (context-enhanced)...")
# Define the second version of your COT prompt.
cot_prompt_v2 = ________

# Loop through each review in the sample, call the generate_output function, and store the results.
cot_v2_list = [generate_output(_________, _________) for review in tqdm(df_sample["Review.Text"], desc="CoT V2 Predictions")]
cot_v2_df = pd.DataFrame(cot_v2_list)
df_sample = pd.concat([df_sample, cot_v2_df.add_prefix("COT_V2_")], axis=1)



In [ ]:
print("Judging CoT V2 outputs...")
# Loop through the generated summaries and use the judge_output function to score them.
cot_v2_scores = [judge_output(_________) for output in tqdm(df_sample["COT_V2_Summary"], desc="Judge COT_V2")]
df_sample["COT_V2_Score"] = cot_v2_scores
print(f"Avg COT_V2_Score: {df_sample['COT_V2_Score'].mean():.3f}")

## **Applying GenAI for Product Recommendation:**

Now, let's use the model for a different task: predicting the Recommended IND flag.

**Questions:**

1.  How do you design a prompt that strictly asks for a binary output (1 or 0) and a brief reason?
    
2.  What kind of function is needed to reliably parse the model's text response to extract the 1/0 flag and the Reason?
    
3.  How do you evaluate the model's performance as a classifier using standard metrics like accuracy, confusion matrix, and classification report?

**How the Process Works**


**1\. Prepare Data**

Copy the dataset, store the original recommendation labels, and remove them from the model input to avoid leakage.

**2\. Generate Predictions**

Use a strict two-line prompt to make the LLM output a binary recommendation (1/0) and a short reason based only on the review text.

**3\. Parse Outputs**

Extract the flag and reason from the raw LLM response using regex-based parsing that handles formatting issues.

**4\. Build Prediction Table**

Run the prompt for each review, parse the result, and store the predictions in a new DataFrame.

 **5\. Evaluate Performance**

Compare LLM predictions with true labels using accuracy, confusion matrix, and classification report.

 **6\. Explain Mismatches**

For incorrect predictions, generate a short explanation describing why the model’s decision may have differed from the human label.

In [ ]:
# --- Predicting Product Recommendations ---
df_reco = df_sample.copy()
df_reco["Actual_Recommended"] = df_reco["Recommended.IND"]
df_reco = df_reco.drop(columns=["Recommended.IND"])

In [ ]:
# The prompt and parsing function are provided for you.
recommendation_prompt = _________


In [ ]:
def parse_recommendation_reply(raw_text):
    if raw_text is None: return (np.nan, "", "")
    s = str(raw_text).strip()
    m = re.search(r'Recommended\s*:\s*([01])\b', s, re.IGNORECASE)
    flag = int(m.group(1)) if m else np.nan
    m3 = re.search(r'Reason\s*:\s*(.+)', s, re.IGNORECASE)
    reason = m3.group(1).strip() if m3 else ""
    return (flag, reason[:200], s)

llm_preds = []
for review in tqdm(df_reco["Review.Text"], desc="LLM Recommend Predictions"):
    prompt = recommendation_prompt + "\n\nReview:\n" + (review or "")
    try:
        # Call the LLM to get a recommendation prediction (1 or 0).
        resp = client.chat.completions.create(
            model=_________,
            messages=[{"role":"user", "content": prompt}],
            temperature=0.0, max_tokens=60
        )
        raw = resp.choices[0].message.content or ""
    except Exception as e:
        raw = f"ERROR: {type(e).__name__}: {e}"

    # Parse the raw response to extract the flag and reason.
    flag, reason, raw_clean = parse_recommendation_reply(raw)
    llm_preds.append({"LLM_Recommended_Raw": raw_clean, "LLM_Recommended_Flag": flag, "LLM_Recommend_Reason": reason})

llm_pred_df = pd.DataFrame(llm_preds)
df_reco = pd.concat([df_reco.reset_index(drop=True), llm_pred_df.reset_index(drop=True)], axis=1)


In [ ]:
# --- Evaluating Model Performance ---
eval_df = df_reco.dropna(subset=["LLM_Recommended_Flag"]).copy()
if not eval_df.empty:
    y_true = eval_df["Actual_Recommended"].astype(int)
    y_pred = eval_df["LLM_Recommended_Flag"].astype(int)

    # Calculate the accuracy score.
    acc = accuracy_score(_________, _________)
    # Generate the confusion matrix.
    cm = confusion_matrix(_________, _________)
    # Generate the classification report.
    report = classification_report(_________, _________, digits=3)

    print(f"Accuracy: {acc:.3f}\n")
    print("Confusion Matrix:\n", cm)
    print("\nClassification report:\n", report)

In [ ]:
# This variable will store the explanation prompt template.
# Later, we will fill in {human_label} and {llm_label} for each mismatched row.
explain_prompt = _______


# -------------------------------------------------------------------------
# STEP 1: Filter all rows where the model's prediction does NOT match the
#         actual human label.
# -------------------------------------------------------------------------
# Explanation:
# - We first make sure the model flag is NOT null.
# - Then we compare the model flag and human flag.
# - Only rows where they differ will be selected for further analysis.
diff_rows = df_reco[
    (df_reco["LLM_Recommended_Flag"].notna()) &
    (df_reco["LLM_Recommended_Flag"].astype(float) != df_reco["Actual_Recommended"].astype(float))
]

# This list will hold the explanation text returned by the LLM for each mismatch
explanations = []


# -------------------------------------------------------------------------
# STEP 2: Loop through each mismatched row and generate an explanation
# -------------------------------------------------------------------------
# NOTE FOR LEARNERS:
# - Using itertuples() is faster and cleaner than iterrows()
# - tqdm() simply adds a progress bar so we can see the loop progress
# -------------------------------------------------------------------------
for row in tqdm(diff_rows.itertuples(), total=len(diff_rows), desc="Explain mismatches"):

    # Safely extract review text (avoid issues if text is empty or None)
    review = getattr(row, "Review_Text", "") or ""

    # Convert values to integers to avoid type-related issues
    llm_label = int(getattr(row, "LLM_Recommended_Flag"))
    human_label = int(getattr(row, "Actual_Recommended"))

    # Build the final prompt by inserting values into the template
    prompt = explain_prompt.format(
        human_label=human_label,
        llm_label=llm_label
    ) + "\n\nReview:\n" + review

    try:
        # -----------------------------------------------------------------
        # STEP 3: Send prompt to the LLM model to generate the explanation
        # -----------------------------------------------------------------
        resp = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "You are a concise analyst."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,       # temperature 0 = more deterministic, consistent outputs
            max_tokens=40          # keep explanation short
        )

        # Extract the text response from the LLM safely
        choice = resp.choices[0]
        msg = getattr(choice, "message", None)

        if msg is not None:
            # Some APIs return a dictionary, others return an object. Handle both.
            raw = msg.get("content", "") if isinstance(msg, dict) else getattr(msg, "content", "") or ""
        elif hasattr(choice, "text"):
            raw = getattr(choice, "text", "") or ""
        else:
            raw = str(choice)

        # Clean the explanation text
        raw = raw.strip().replace("\n", " ")

        # Store a trimmed version of the explanation to keep things neat
        explanations.append(raw[:250])

    except Exception as e:
        # If anything fails (rate limit, API error), store an error message instead
        explanations.append(f"ERROR: {type(e).__name__}")


# -------------------------------------------------------------------------
# STEP 4: Add a new column to df_reco for storing explanations
# -------------------------------------------------------------------------
# IMPORTANT:
# - Setting dtype to object ensures pandas accepts string entries
df_reco["LLM_Diff_Explain"] = pd.Series(dtype='object')


# -------------------------------------------------------------------------
# STEP 5: Attach each explanation to the correct row using index alignment
# -------------------------------------------------------------------------
for i, (idx, row) in enumerate(diff_rows.iterrows()):
    df_reco.at[idx, "LLM_Diff_Explain"] = explanations[i]


# Final confirmation message
print(f"Added explanations for {len(diff_rows)} mismatches.")

df_reco


**Visualization of Sentiments Distribution**

 After generating results from all prompting techniques, it's crucial to visualize their outputs to better understand their behavior and performance. This helps us see if one technique tends to be more cautious (e.g., assigning more 'Neutral' sentiments) or if they generally agree on the sentiment of the reviews.
    
 **Questions:**
    
* How does the distribution of predicted Sentiment (Positive, Negative, Neutral) compare across the V2 versions of Zero-Shot, Few-Shot, and Chain-of-Thought? (Hint: Create a separate bar chart for each technique's V2 sentiment column).
    
* Are there noticeable differences in the counts? For example, does one technique identify more "Neutral" reviews than the others? What might this imply about its ability to handle nuance?
    


In [ ]:

# ----------------------------
# Helper function for bar chart
# ----------------------------
def plot_sentiment(df, column_name, title):
    sentiment_counts = df[column_name].value_counts().reset_index()
    sentiment_counts.columns = ["Sentiment", "Count"]

    fig = px.bar(
        sentiment_counts,
        x="Sentiment",
        y="Count",
        text="Count",
        title=title,
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(xaxis_title="Sentiment", yaxis_title="Count")
    fig.show()


# ----------------------------
# Plot all three methods
# ----------------------------

# ----------------------------
# Plot all three methods
# ----------------------------
# Learner Notes:
# Replace these column names with the exact ones from your dataset.
# Below are common names used in sentiment evaluation for V1 and V2.
# ----------------------------



plot_sentiment(df_sample, _________, "Zero-Shot Sentiment Distribution for Version 1")
plot_sentiment(df_sample, _________, "Few-Shot Sentiment Distribution for Version 1")
plot_sentiment(df_sample, _________, "Chain-of-Thought Sentiment Distribution for Version 1")
plot_sentiment(df_sample, _________, "Zero-Shot Sentiment Distribution for Version 2")
plot_sentiment(df_sample,_________, "Few-Shot Sentiment Distribution for Version 2")
plot_sentiment(df_sample, _________, "Chain-of-Thought Sentiment Distribution for Version 2")



##  **Comparison of Prompting Techniques:**
    
   *   How do the three techniques (Zero-Shot, Few-Shot, CoT) compare in terms of their responses. Use LLM to give verdict?
        
  *   Which technique was the most reliable and consistent? Why do you think it performed the best?
        
   *   What model and prompt design would you propose for a production environment?
        


In [ ]:
# ---------------------------------------------------------
# Build comparison records for the LLM to analyze
# ---------------------------------------------------------

comparison_records = []
for idx, row in df_sample.iterrows():
    rec = {
        "Review": row.get("Review.Text", ""),              # Here we pull the sentiment from each technique. Replace these with V2 Sentiments of each techniques.
        "ZeroShot_Sentiment": row.get(________, None),
        "FewShot_Sentiment": row.get(________, None),
        "CoT_Sentiment": row.get(________, None)
    }
    comparison_records.append(rec)
summary_text = json.dumps(comparison_records, indent=2)

compare_prompt = ___________

# Call the LLM with the compare_prompt to get a qualitative comparison of the techniques.
response = client.chat.completions.create(
    model=_________,
    messages=[
        {"role": "system", "content": "You are an expert in evaluating LLM prompting strategies."},
        {"role": "user", "content": compare_prompt}
    ],
    temperature=0
)
print(response.choices[0].message.content)

### **Observations and Insights**

 **Refined Insights:**
    
   *   What are the most meaningful and recurring insights from the customer reviews, as identified by your best-performing model?

# Generating Actionable Product Improvement Suggestions


 *   Based on the aggregated insights from your best model, what are 3 short-term (3-6 months) and 3 long-term (6-12 months) actionable business recommendations for the retail company?
        
 *   How does this automated GenAI pipeline solve the initial business problem and create value?

In [ ]:
business_prompt = _____________

# Call the LLM with the business_prompt to generate strategic recommendations.
response = client.chat.completions.create(
    model=_________,
    messages=[
        {"role": "system", "content": "You are a retail business strategist."},
        {"role": "user", "content": business_prompt}
    ],
    temperature=0
)
print(response.choices[0].message.content)

### **Observations and Insights**

## **Conclusion**